---
#  Vectorization #2 
---

In [1]:
%load_ext autoreload
%autoreload 2

import visual, clustering, tests, reference

---
## K-means Clustering
---
Clustering is an unsupervised subclass of machine learning. While most deep learning applications use data $x_i$ with label $y_i$, unsupervised algorithms only work with data. Clustering is the task to partition the dataset into clusters, based on some kind of similarity. We will look at the most natural clustering algorithm, which is K-means (or more specifically, Lloyd's algorithm to solve the K-means problem). K-means groups the data into $K$ different clusters, which are defined by their cluster centroid (a.k.a. _cluster centers_). A data point belongs to the cluster with the lowest Euclidean distance to the corresponding centroid.  The algorithm works as follows:

1. Initialize the cluster centers. (We provide a clever initalization scheme, you don't have to implement this yourself.)
2. Repeat until convergence:
   1. E-step: Assign all data points to their nearest cluster center
   2. M-step: Recalculate the cluster centers as mean of the points belonging to the cluster

```
def k_means(X: torch.Tensor, k: int):
    N, D = X.shape
    centers = kmeans_plusplus(X, k)  # k x D

    # calculate initial cluster assignments
    assignments = torch.zeros(N, dtype=torch.long)
    for n in range(N):
        min_dist = torch.inf
        min_idx = -1
        for j in range(k):
            dist = torch.sqrt(torch.sum((X[n] - centers[j]) ** 2))
            if dist < min_dist:
                min_dist = dist
                min_idx = j

        assignments[n] = min_idx

    for i in range(100): # Limit the maximum number of loop iterations to avoid livelock
        # M step: update the cluster centers to be the mean of its elements
        for j in range(k):
            sum = torch.zeros(D)
            num_elements = 0
            for n in range(N):
                if assignments[n] == j:
                    sum += X[n]
                    num_elements += 1
            # only update centers that have elements (div by 0!), otherwise ignore
            if num_elements > 0:
                centers[j] = sum / num_elements

        # E step: calculate which center has the lowest distance to all data points
        new_assignments = torch.empty_like(assignments)
        for n in range(N):
            min_dist = torch.inf
            min_idx = -1
            for j in range(k):
                dist = torch.sqrt(torch.sum((X[n] - centers[j]) ** 2))
                if dist < min_dist:
                    min_dist = dist
                    min_idx = j

            new_assignments[n] = min_idx

        # check for convergance
        if torch.all(assignments == new_assignments):
            break
        assignments = new_assignments

    return assignments, centers, i
```
---

## **Task**: 

Implement `k_means_vec` in `clustering.py` by vectorizing (most) loops. Take a look at `k_means` for reference. Run the code below to check the implementation.


**Hints:**

- You can't vectorize all loops from `k_means`. Think about what needs to happen sequentially and which calculations can be done at the same time.
- To vectorize the distance calculation, think about how to modify the pairwise distances from the problem before.
- To find the smallest distances, you can use `torch.argmin`.

First, check how the naive implementation clusters the data:

In [2]:
X = tests.sample_data(N=100)
a, c, i = reference.k_means(X, k=5)
print(f"converged in {i} iterations")
tests.check_cluster_valid(X, c, a)
visual.plot_clusters(X, c, a, "Clusters in Reference Implementation").show()

converged in 4 iterations
The clusters are a valid solution


Run the code here to check the vectorized implementation. This cell uses the same data as above. To sample new points rerun the cell above.

In [13]:
a_vec, c_vec, i_vec = clustering.k_means_vec(X, k=5)
print(f"converged in {i} iterations")
tests.check_cluster_valid(X, c_vec, a_vec)
visual.plot_clusters(X, c_vec, a_vec, "Vectorized Clusters")

converged in 7 iterations
The clusters are a valid solution


---

### Test 3D Spherical Data

You can also check how your algorithm performs on data living on a spherical manifold. If your algorithm works correctly, the result should look like continents on a globe.

In [14]:
X = tests.sample_data(N=1_000, D=3, hypersphere=True)
a, c, i = clustering.k_means_vec(X, k=7)
print(f"Converged in {i} iterations")
tests.check_cluster_valid(X, c, a)
visual.plot_clusters_3d(X, c, a, "Spherical Data Clustering").show()

Converged in 9 iterations
The clusters are a valid solution


---

### Speed Test

Once again, we will check the speed of our implementation. The speedup should be about ~10-100 times.

In [15]:
tests.benchmark_kmeans().show()

---
## K-means (again, harder)
---
In your previous solution, you (probably) calculated the cluster centers one at a time. It is actually possible to vectorize this operation via _scatter-reduce_.

Scatter-reduce offers a way to group elements and combine ("reduce") them in a single operation. The select elements are specified by an `index` tensor, which selects the elements from a `src` data tensor. Typical reductions are sum, mean, etc.

Look at this example, where a sum reduction is performed:

<img alt="Scatter Reduce" src="PDL_scatter_reduce.png"/>

In the current pytorch implementations, an `input` tensor has to be provided in which the result is written. This tensor can be part of the reduction based on the `include_self` parameter.

Note that the `index` and `src` tensor are not broadcasted, meaning that if you work with multidimensional data, you will have to provide an index for each dimension, e.g. if you want to add $n$ vectors of dimension 2, you need to provide an $n \times 2$ `index` vector. If the index is the same along multiple dimensions, `tensor.repeat_interleave` is one possible option to create such a tensor.

---

## **Task**: 

Implement `k_means_scatter` in `clustering.py` by using `tensor.scatter_reduce`. Run the code here to check the implementation.

**Hints:**
- There are no tricks here, you just need to tinker a bit to get all parts to work correctly.


In [7]:
X = tests.sample_data(N=25_000, D=2)
a, c, i = clustering.k_means_scatter(X, k=7)
print(f"converged in {i} iterations")
tests.check_cluster_valid(X, c, a)
visual.plot_clusters(X, c, a, "Clusters").show()

converged in 89 iterations
The clusters are a valid solution


---

### Speed Test

Once again, we will check the speed of our implementation. The speedup should be about ~2 times for $K = 25$ Clusters, but the advantage will grow with the number of Clusters.

In [5]:
tests.benchmark_kmeans_scatter().show()